### Import packages ###

In [1]:

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import scipy.stats as stats
from sklearn.preprocessing import StandardScaler,MinMaxScaler,RobustScaler
from sklearn.metrics import classification_report,confusion_matrix
from sklearn.model_selection import GridSearchCV,train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier,RandomForestClassifier
from sklearn.impute import KNNImputer,SimpleImputer
from sklearn.feature_selection import RFE,SelectKBest,f_classif,f_regression
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.ensemble import AdaBoostClassifier,GradientBoostingClassifier,VotingClassifier,RandomForestClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from imblearn.over_sampling import SMOTE
from sklearn.svm import SVC

from sqlalchemy import create_engine
import joblib

### Import datasets ###

In [2]:
server=r'DESKTOP-D627T83\SQLEXPRESS'
database='INTERNSHIP'

connection_string=f"mssql+pyodbc://@{server}/{database}?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"

engine=create_engine(connection_string)

In [3]:
query1="select * from orders"
orders=pd.read_sql(query1,con=engine)

query2="select * from customer"
customer=pd.read_sql(query2,con=engine)

query3="SELECT geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,TRANSLATE(geolocation_city,\
'áàâãäéèêëíìîïóòôõöúùûüçÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇ','aaaaaeeeeiiiiooooouuuucAAAAAEEEEIIIIOOOOOUUUUC') \
AS geolocation_city,geolocation_state FROM geolocation_temp"
geolocation=pd.read_sql(query3,con=engine)

query4="select * from order_items"
order_items=pd.read_sql(query4,con=engine)

query5="select * from order_payments_clean"
order_payments=pd.read_sql(query5,con=engine)

query6="select * from order_reviews_clean"
order_reviews=pd.read_sql(query6,con=engine)

query7="select * from product_category_name_translation"
product_category=pd.read_sql(query7,con=engine)

query8="select * from products_clean"
products=pd.read_sql(query8,con=engine)

query9="select * from sellers"
sellers=pd.read_sql(query9,con=engine)

In [4]:
geolocation_new=geolocation.groupby('geolocation_zip_code_prefix')[['geolocation_lat','geolocation_lng','geolocation_city','geolocation_state']].agg(
    geolocation_lat=('geolocation_lat','mean'),
    geolocation_lng=('geolocation_lng','mean'),
    geolocation_city=('geolocation_city','max'),
    geolocation_state=('geolocation_state','max')).reset_index()

### Data Audit ###

In [5]:
customer.head(3)

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,00012a2ce6f8dcda20d059ce98491703,248ffe10d632bebe4f7267f1f44844c9,6273,osasco,SP
1,000161a058600d5901f007fab4c27140,b0015e09bb4b6e47c52844fab5fb6638,35550,itapecerica,MG
2,0001fd6190edaaf884bcaf3d49edf079,94b11d37cd61cb2994a194d11f89682b,29830,nova venecia,ES


In [6]:
customer.shape

(99441, 5)

In [7]:
customer.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  int64
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: int64(1), str(4)
memory usage: 11.0 MB


In [8]:
# Check for missing values

customer.isna().sum()

customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

In [9]:
orders.head(3)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,48fcb88788cb3df0296a5dcf8f95ac68,fffb97495f78be80e2759335275df2aa,delivered,2018-01-16 14:51:00,2018-01-16 15:09:00,2018-01-17 18:18:00,2018-01-26 20:16:00,2018-02-06
1,2db1e25288f9a89e2ed181d8b6216f4a,0d3fdbd4697ff7426b3d0fbebafa60c1,delivered,2018-04-11 23:17:00,2018-04-13 02:31:00,2018-04-19 15:38:00,2018-04-30 11:22:00,2018-05-09
2,5b018cb93699b67367756f17e0a383a8,b8f6934b2c3cc47158ec691d288b04f0,delivered,2017-09-25 15:21:00,2017-09-25 15:35:00,2017-09-26 18:54:00,2017-10-02 16:49:00,2017-10-23


In [10]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  str           
 1   customer_id                    99441 non-null  str           
 2   order_status                   99441 non-null  str           
 3   order_purchase_timestamp       99441 non-null  datetime64[us]
 4   order_approved_at              99295 non-null  datetime64[us]
 5   order_delivered_carrier_date   97660 non-null  datetime64[us]
 6   order_delivered_customer_date  96484 non-null  datetime64[us]
 7   order_estimated_delivery_date  99441 non-null  datetime64[us]
dtypes: datetime64[us](5), str(3)
memory usage: 13.0 MB


### To treat missing values, in this model we will consider orders which are delivered ###

In [11]:
orders_new=orders.loc[orders['order_status']=='delivered']
orders_new

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,48fcb88788cb3df0296a5dcf8f95ac68,fffb97495f78be80e2759335275df2aa,delivered,2018-01-16 14:51:00,2018-01-16 15:09:00,2018-01-17 18:18:00,2018-01-26 20:16:00,2018-02-06
1,2db1e25288f9a89e2ed181d8b6216f4a,0d3fdbd4697ff7426b3d0fbebafa60c1,delivered,2018-04-11 23:17:00,2018-04-13 02:31:00,2018-04-19 15:38:00,2018-04-30 11:22:00,2018-05-09
2,5b018cb93699b67367756f17e0a383a8,b8f6934b2c3cc47158ec691d288b04f0,delivered,2017-09-25 15:21:00,2017-09-25 15:35:00,2017-09-26 18:54:00,2017-10-02 16:49:00,2017-10-23
3,35147acb423f303e15d12cfa60292799,1be97ded06a3b15e8bc7d21641c7dcb0,delivered,2018-04-18 18:13:00,2018-04-19 02:31:00,2018-04-19 21:46:00,2018-04-27 21:38:00,2018-05-15
4,c378d1766ed412f14e38a4fd8547f9e9,4b0dbdea9ec36d6a05622ae783368715,delivered,2017-10-05 11:59:00,2017-10-05 12:07:00,2017-10-05 17:22:00,2017-10-09 17:38:00,2017-10-24
...,...,...,...,...,...,...,...,...
99432,580f3268bc2075c5961b2d2929c7a35b,ce37390d21ccd85e25c90ba634097c29,delivered,2017-09-01 20:17:00,2017-09-13 22:03:00,2017-09-15 18:03:00,2017-09-18 17:52:00,2017-09-25
99433,0faf27f68362829970b17e9b0989b20d,dd1239e9f1ec85658411937ae783f6c7,delivered,2018-08-09 22:53:00,2018-08-09 23:05:00,2018-08-10 14:07:00,2018-08-15 15:33:00,2018-08-23
99435,1756895428025c2821aed95304b24fab,5a90682cd27ad9775778b346b7d53044,delivered,2018-01-22 16:15:00,2018-01-23 03:38:00,2018-01-24 19:19:00,2018-02-05 15:37:00,2018-02-21
99438,f9cda286ca417062ff15bf5d63fbc70c,7b2a178190dfc4efadeccb75fc833074,delivered,2018-05-21 17:25:00,2018-05-21 17:56:00,2018-05-22 10:49:00,2018-06-06 20:37:00,2018-06-11


In [12]:
orders_new.isna().sum()

order_id                         0
customer_id                      0
order_status                     0
order_purchase_timestamp         0
order_approved_at                0
order_delivered_carrier_date     0
order_delivered_customer_date    0
order_estimated_delivery_date    0
dtype: int64

In [13]:
geolocation_new.head(3)

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1001,-23.550271,-46.634047,sao paulo,SP
1,1002,-23.547657,-46.634991,sao paulo,SP
2,1003,-23.548991,-46.635653,sao paulo,SP


In [14]:
geolocation_new.info()

<class 'pandas.DataFrame'>
RangeIndex: 19015 entries, 0 to 19014
Data columns (total 5 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   geolocation_zip_code_prefix  19015 non-null  int64  
 1   geolocation_lat              19015 non-null  float64
 2   geolocation_lng              19015 non-null  float64
 3   geolocation_city             19015 non-null  str    
 4   geolocation_state            19015 non-null  str    
dtypes: float64(2), int64(1), str(2)
memory usage: 975.6 KB


In [15]:
geolocation_new.nunique()

geolocation_zip_code_prefix    19015
geolocation_lat                19001
geolocation_lng                19001
geolocation_city                5757
geolocation_state                 27
dtype: int64

In [16]:
order_items.head(3)

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:00,58.900002,13.290000
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:00,239.899994,19.930000
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:00,199.000000,17.870001


In [17]:
order_items.info()

<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   order_id             112650 non-null  str           
 1   order_item_id        112650 non-null  int64         
 2   product_id           112650 non-null  str           
 3   seller_id            112650 non-null  str           
 4   shipping_limit_date  112650 non-null  datetime64[us]
 5   price                112650 non-null  float64       
 6   freight_value        112650 non-null  float64       
dtypes: datetime64[us](1), float64(2), int64(1), str(3)
memory usage: 16.3 MB


In [18]:
order_payments.head(3)

,order_id,tot_orders,Total_installments,credit_card,boleto,voucher,debit_card,others,total_amount
0,00010242fe8c5a6d1ba2dd792cb16214,1,2,72.19,0.0,0.0,0.0,0.0,72.19
1,00018f77f2f0320c557190d7a144bdd3,1,3,259.83,0.0,0.0,0.0,0.0,259.83
2,000229ec398224ef6ca0657da4fc703e,1,5,216.87,0.0,0.0,0.0,0.0,216.87


In [19]:
order_payments.info()

<class 'pandas.DataFrame'>
RangeIndex: 99440 entries, 0 to 99439
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   order_id            99440 non-null  str    
 1   tot_orders          99440 non-null  int64  
 2   Total_installments  99440 non-null  int64  
 3   credit_card         99440 non-null  float64
 4   boleto              99440 non-null  float64
 5   voucher             99440 non-null  float64
 6   debit_card          99440 non-null  float64
 7   others              99440 non-null  float64
 8   total_amount        99440 non-null  float64
dtypes: float64(6), int64(2), str(1)
memory usage: 9.9 MB


In [20]:
order_reviews.head(3)

,order_id,review_id,review_score,review_creation_date,review_answer_timestamp
0,00010242fe8c5a6d1ba2dd792cb16214,97ca439bc427b48bc1cd7177abe71365,5,2017-09-21,2017-09-22 10:57:00
1,00018f77f2f0320c557190d7a144bdd3,7b07bacd811c4117b742569b04ce3580,4,2017-05-13,2017-05-15 11:34:00
2,000229ec398224ef6ca0657da4fc703e,0c5b33dea94867d1ac402749e5438e8b,5,2018-01-23,2018-01-23 16:06:00


In [21]:
order_reviews.info()

<class 'pandas.DataFrame'>
RangeIndex: 98673 entries, 0 to 98672
Data columns (total 5 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   order_id                 98673 non-null  str           
 1   review_id                98673 non-null  str           
 2   review_score             98673 non-null  int64         
 3   review_creation_date     98673 non-null  datetime64[us]
 4   review_answer_timestamp  98673 non-null  datetime64[us]
dtypes: datetime64[us](2), int64(1), str(2)
memory usage: 9.8 MB


In [22]:
product_category.head(3)

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto


In [23]:
product_category.info()

<class 'pandas.DataFrame'>
RangeIndex: 71 entries, 0 to 70
Data columns (total 2 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   product_category_name          71 non-null     str  
 1   product_category_name_english  71 non-null     str  
dtypes: str(2)
memory usage: 3.5 KB


In [24]:
products.head(3)

,product_id,product_category_name,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,00066f42aeeb9f3007548bb9d3f33c38,perfumaria,6,300.0,20.0,16.0,16.0
1,00088930e925c41fd95ebfe695fd2655,automotivo,4,1225.0,55.0,10.0,26.0
2,0009406fd7479715e4bef61dd91f2462,cama_mesa_banho,2,300.0,45.0,15.0,35.0


In [25]:
products.info()

<class 'pandas.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 7 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   product_id             32951 non-null  str    
 1   product_category_name  32951 non-null  str    
 2   product_photos_qty     32951 non-null  int64  
 3   product_weight_g       32949 non-null  float64
 4   product_length_cm      32949 non-null  float64
 5   product_height_cm      32949 non-null  float64
 6   product_width_cm       32949 non-null  float64
dtypes: float64(4), int64(1), str(2)
memory usage: 3.2 MB


In [26]:
products.isna().sum()

product_id               0
product_category_name    0
product_photos_qty       0
product_weight_g         2
product_length_cm        2
product_height_cm        2
product_width_cm         2
dtype: int64

In [27]:
sellers.head(3)

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,0015a82c2db000af6aaaf3ae2ecb0532,9080,santo andre,SP
1,001cca7ae9ae17fb1caed9dfb1094831,29156,cariacica,ES
2,001e6ad469a905060d959994f1b41e4f,24754,sao goncalo,RJ


In [28]:
sellers.info()

<class 'pandas.DataFrame'>
RangeIndex: 3095 entries, 0 to 3094
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   seller_id               3095 non-null   str  
 1   seller_zip_code_prefix  3095 non-null   int64
 2   seller_city             3095 non-null   str  
 3   seller_state            3095 non-null   str  
dtypes: int64(1), str(3)
memory usage: 230.3 KB


## Calculated fields ##

In [29]:
order_reviews['review_answer_time']=(order_reviews['review_answer_timestamp']-order_reviews['review_creation_date']).dt.days

In [30]:
order_reviews_df=order_reviews[['order_id','review_score','review_answer_time']]
order_reviews_df.head(3)

,order_id,review_score,review_answer_time
0,00010242fe8c5a6d1ba2dd792cb16214,5,1
1,00018f77f2f0320c557190d7a144bdd3,4,2
2,000229ec398224ef6ca0657da4fc703e,5,0


In [31]:
order_item_df=order_items.groupby(['order_id','product_id'])[['order_item_id','price','freight_value']].agg(total_items=('order_item_id','count'),
                                                                                              total_price=('price','sum'),
                                                                                              total_freight_value=('freight_value','sum')).reset_index()
order_item_df

,order_id,product_id,total_items,total_price,total_freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,4244733e06e7ecb4970a6e2683c13e61,1,58.900002,13.290000
1,00018f77f2f0320c557190d7a144bdd3,e5f2d52b802189ee658865ca93d83a8f,1,239.899994,19.930000
2,000229ec398224ef6ca0657da4fc703e,c777355d18b72b67abbeef9df44fd0fd,1,199.000000,17.870001
3,00024acbcdf0a6daa1e931b038114c75,7634da152a4610f1595efa32f14722fc,1,12.990000,12.790000
4,00042b26cf59d7ce69dfabb4e55b4fd9,ac6c3623068f30de03045865e4e10089,1,199.899994,18.139999
...,...,...,...,...,...
102420,fffc94f6ce00a00581880bf54a75a037,4aa6014eceb682077f9dc4bffebc05b0,1,299.989990,43.410000
102421,fffcd46ef2263f404302a634eb57f7eb,32e07fd915822b0765e448c4dd74c828,1,350.000000,36.529999
102422,fffce4705a9662cd70adb13d4a31832d,72a30483855e2eafc67aee5dc2560482,1,99.900002,16.950001
102423,fffe18544ffabc95dfada21779c9644f,9c422a519119dcad7575db5af1ba540e,1,55.990002,8.720000


In [32]:
orders_new['order_approval_hours']=(orders_new['order_approved_at']-orders_new['order_purchase_timestamp']).dt.total_seconds()/3600
orders_new['carrier_delivered_days']=(orders_new['order_delivered_carrier_date']-orders_new['order_approved_at']).dt.days
orders_new['customer_delivered_days']=(orders_new['order_delivered_customer_date']-orders_new['order_delivered_carrier_date']).dt.days
orders_new['order_timely_delivered?']=np.where(orders_new['order_estimated_delivery_date']>=orders_new['order_delivered_customer_date'],1,0)
orders_new['extra_days_taken_for_delivery']=(orders_new['order_delivered_customer_date']-orders_new['order_estimated_delivery_date']).dt.days

In [33]:
orders_df=orders_new[['order_id','customer_id','order_approval_hours','carrier_delivered_days',
            'customer_delivered_days','order_timely_delivered?','extra_days_taken_for_delivery']]
orders_df

,order_id,customer_id,order_approval_hours,carrier_delivered_days,customer_delivered_days,order_timely_delivered?,extra_days_taken_for_delivery
0,48fcb88788cb3df0296a5dcf8f95ac68,fffb97495f78be80e2759335275df2aa,0.300000,1,9,1,-11
1,2db1e25288f9a89e2ed181d8b6216f4a,0d3fdbd4697ff7426b3d0fbebafa60c1,27.233333,6,10,1,-9
2,5b018cb93699b67367756f17e0a383a8,b8f6934b2c3cc47158ec691d288b04f0,0.233333,1,5,1,-21
3,35147acb423f303e15d12cfa60292799,1be97ded06a3b15e8bc7d21641c7dcb0,8.300000,0,7,1,-18
4,c378d1766ed412f14e38a4fd8547f9e9,4b0dbdea9ec36d6a05622ae783368715,0.133333,0,4,1,-15
...,...,...,...,...,...,...,...
99432,580f3268bc2075c5961b2d2929c7a35b,ce37390d21ccd85e25c90ba634097c29,289.766667,1,2,1,-7
99433,0faf27f68362829970b17e9b0989b20d,dd1239e9f1ec85658411937ae783f6c7,0.200000,0,5,1,-8
99435,1756895428025c2821aed95304b24fab,5a90682cd27ad9775778b346b7d53044,11.383333,1,11,1,-16
99438,f9cda286ca417062ff15bf5d63fbc70c,7b2a178190dfc4efadeccb75fc833074,0.516667,0,15,1,-5


In [34]:
order_payments

,order_id,tot_orders,Total_installments,credit_card,boleto,voucher,debit_card,others,total_amount
0,00010242fe8c5a6d1ba2dd792cb16214,1,2,72.19,0.00,0.0,0.0,0.0,72.19
1,00018f77f2f0320c557190d7a144bdd3,1,3,259.83,0.00,0.0,0.0,0.0,259.83
2,000229ec398224ef6ca0657da4fc703e,1,5,216.87,0.00,0.0,0.0,0.0,216.87
3,00024acbcdf0a6daa1e931b038114c75,1,2,25.78,0.00,0.0,0.0,0.0,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,3,218.04,0.00,0.0,0.0,0.0,218.04
...,...,...,...,...,...,...,...,...,...
99435,fffc94f6ce00a00581880bf54a75a037,1,1,0.00,343.40,0.0,0.0,0.0,343.40
99436,fffcd46ef2263f404302a634eb57f7eb,1,1,0.00,386.53,0.0,0.0,0.0,386.53
99437,fffce4705a9662cd70adb13d4a31832d,1,3,116.85,0.00,0.0,0.0,0.0,116.85
99438,fffe18544ffabc95dfada21779c9644f,1,3,64.71,0.00,0.0,0.0,0.0,64.71


In [35]:
products=products.dropna()
products

,product_id,product_category_name,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,00066f42aeeb9f3007548bb9d3f33c38,perfumaria,6,300.0,20.0,16.0,16.0
1,00088930e925c41fd95ebfe695fd2655,automotivo,4,1225.0,55.0,10.0,26.0
2,0009406fd7479715e4bef61dd91f2462,cama_mesa_banho,2,300.0,45.0,15.0,35.0
3,000b8f95fcb9e0096488278317764d19,utilidades_domesticas,3,550.0,19.0,24.0,12.0
4,000d9be29b5207b54e86aa1b1ac54872,relogios_presentes,4,250.0,22.0,11.0,15.0
...,...,...,...,...,...,...,...
32946,fff6177642830a9a94a0f2cba5e476d1,cool_stuff,1,1400.0,42.0,25.0,15.0
32947,fff81cc3158d2725c0655ab9ba0f712c,beleza_saude,1,58.0,16.0,16.0,11.0
32948,fff9553ac224cec9d15d49f5a263411f,fashion_bolsas_e_acessorios,3,150.0,16.0,13.0,11.0
32949,fffdb2d0ec8d6a61f0a0a0db3f25b441,informatica_acessorios,5,150.0,16.0,4.0,13.0


In [36]:
products

,product_id,product_category_name,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,00066f42aeeb9f3007548bb9d3f33c38,perfumaria,6,300.0,20.0,16.0,16.0
1,00088930e925c41fd95ebfe695fd2655,automotivo,4,1225.0,55.0,10.0,26.0
2,0009406fd7479715e4bef61dd91f2462,cama_mesa_banho,2,300.0,45.0,15.0,35.0
3,000b8f95fcb9e0096488278317764d19,utilidades_domesticas,3,550.0,19.0,24.0,12.0
4,000d9be29b5207b54e86aa1b1ac54872,relogios_presentes,4,250.0,22.0,11.0,15.0
...,...,...,...,...,...,...,...
32946,fff6177642830a9a94a0f2cba5e476d1,cool_stuff,1,1400.0,42.0,25.0,15.0
32947,fff81cc3158d2725c0655ab9ba0f712c,beleza_saude,1,58.0,16.0,16.0,11.0
32948,fff9553ac224cec9d15d49f5a263411f,fashion_bolsas_e_acessorios,3,150.0,16.0,13.0,11.0
32949,fffdb2d0ec8d6a61f0a0a0db3f25b441,informatica_acessorios,5,150.0,16.0,4.0,13.0


## Combine tables ##

In [37]:
df2=pd.merge(orders_df,order_payments,on='order_id',how='inner')
df2

,order_id,customer_id,order_approval_hours,carrier_delivered_days,customer_delivered_days,order_timely_delivered?,extra_days_taken_for_delivery,tot_orders,Total_installments,credit_card,boleto,voucher,debit_card,others,total_amount
0,48fcb88788cb3df0296a5dcf8f95ac68,fffb97495f78be80e2759335275df2aa,0.300000,1,9,1,-11,1,3,61.01,0.00,0.0,0.0,0.0,61.01
1,2db1e25288f9a89e2ed181d8b6216f4a,0d3fdbd4697ff7426b3d0fbebafa60c1,27.233333,6,10,1,-9,1,1,0.00,222.20,0.0,0.0,0.0,222.20
2,5b018cb93699b67367756f17e0a383a8,b8f6934b2c3cc47158ec691d288b04f0,0.233333,1,5,1,-21,1,5,245.35,0.00,0.0,0.0,0.0,245.35
3,35147acb423f303e15d12cfa60292799,1be97ded06a3b15e8bc7d21641c7dcb0,8.300000,0,7,1,-18,1,1,0.00,37.13,0.0,0.0,0.0,37.13
4,c378d1766ed412f14e38a4fd8547f9e9,4b0dbdea9ec36d6a05622ae783368715,0.133333,0,4,1,-15,1,2,21.77,0.00,0.0,0.0,0.0,21.77
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96472,580f3268bc2075c5961b2d2929c7a35b,ce37390d21ccd85e25c90ba634097c29,289.766667,1,2,1,-7,1,12,122.10,0.00,0.0,0.0,0.0,122.10
96473,0faf27f68362829970b17e9b0989b20d,dd1239e9f1ec85658411937ae783f6c7,0.200000,0,5,1,-8,1,8,816.89,0.00,0.0,0.0,0.0,816.89
96474,1756895428025c2821aed95304b24fab,5a90682cd27ad9775778b346b7d53044,11.383333,1,11,1,-16,1,1,0.00,176.87,0.0,0.0,0.0,176.87
96475,f9cda286ca417062ff15bf5d63fbc70c,7b2a178190dfc4efadeccb75fc833074,0.516667,0,15,1,-5,1,1,24.13,0.00,0.0,0.0,0.0,24.13


In [38]:
df3=pd.merge(df2,customer[['customer_id','customer_zip_code_prefix']],on='customer_id',how='inner')
df3

,order_id,customer_id,order_approval_hours,carrier_delivered_days,customer_delivered_days,order_timely_delivered?,extra_days_taken_for_delivery,tot_orders,Total_installments,credit_card,boleto,voucher,debit_card,others,total_amount,customer_zip_code_prefix
0,48fcb88788cb3df0296a5dcf8f95ac68,fffb97495f78be80e2759335275df2aa,0.300000,1,9,1,-11,1,3,61.01,0.00,0.0,0.0,0.0,61.01,31140
1,2db1e25288f9a89e2ed181d8b6216f4a,0d3fdbd4697ff7426b3d0fbebafa60c1,27.233333,6,10,1,-9,1,1,0.00,222.20,0.0,0.0,0.0,222.20,35290
2,5b018cb93699b67367756f17e0a383a8,b8f6934b2c3cc47158ec691d288b04f0,0.233333,1,5,1,-21,1,5,245.35,0.00,0.0,0.0,0.0,245.35,24322
3,35147acb423f303e15d12cfa60292799,1be97ded06a3b15e8bc7d21641c7dcb0,8.300000,0,7,1,-18,1,1,0.00,37.13,0.0,0.0,0.0,37.13,88780
4,c378d1766ed412f14e38a4fd8547f9e9,4b0dbdea9ec36d6a05622ae783368715,0.133333,0,4,1,-15,1,2,21.77,0.00,0.0,0.0,0.0,21.77,4536
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96472,580f3268bc2075c5961b2d2929c7a35b,ce37390d21ccd85e25c90ba634097c29,289.766667,1,2,1,-7,1,12,122.10,0.00,0.0,0.0,0.0,122.10,88352
96473,0faf27f68362829970b17e9b0989b20d,dd1239e9f1ec85658411937ae783f6c7,0.200000,0,5,1,-8,1,8,816.89,0.00,0.0,0.0,0.0,816.89,14610
96474,1756895428025c2821aed95304b24fab,5a90682cd27ad9775778b346b7d53044,11.383333,1,11,1,-16,1,1,0.00,176.87,0.0,0.0,0.0,176.87,93336
96475,f9cda286ca417062ff15bf5d63fbc70c,7b2a178190dfc4efadeccb75fc833074,0.516667,0,15,1,-5,1,1,24.13,0.00,0.0,0.0,0.0,24.13,85660


In [39]:
df4=pd.merge(order_reviews_df,df3,on='order_id',how='inner')
df4

,order_id,review_score,review_answer_time,customer_id,order_approval_hours,carrier_delivered_days,customer_delivered_days,order_timely_delivered?,extra_days_taken_for_delivery,tot_orders,Total_installments,credit_card,boleto,voucher,debit_card,others,total_amount,customer_zip_code_prefix
0,00010242fe8c5a6d1ba2dd792cb16214,5,1,3ce436f183e68e07877b285a838db11a,0.766667,6,1,1,-9,1,2,72.19,0.00,0.0,0.0,0.0,72.19,28013
1,00018f77f2f0320c557190d7a144bdd3,4,2,f6dd3ec061db4e3987629fe6b26e5cce,0.200000,8,8,1,-3,1,3,259.83,0.00,0.0,0.0,0.0,259.83,15775
2,000229ec398224ef6ca0657da4fc703e,5,0,6489ae5e4333f3693df5ad4372dab6d3,0.250000,1,6,1,-14,1,5,216.87,0.00,0.0,0.0,0.0,216.87,35661
3,00024acbcdf0a6daa1e931b038114c75,4,0,d4eb9395c8c0431ee92fce09860c5a06,0.166667,2,4,1,-6,1,2,25.78,0.00,0.0,0.0,0.0,25.78,12952
4,00042b26cf59d7ce69dfabb4e55b4fd9,5,1,58dbd0b2d70206bf40e62cd34e84d795,0.216667,11,13,1,-16,1,3,218.04,0.00,0.0,0.0,0.0,218.04,13226
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95826,fffc94f6ce00a00581880bf54a75a037,5,3,b51593916b4b8e0d6f66f2ae24f2673d,38.233333,0,15,1,-8,1,1,0.00,343.40,0.0,0.0,0.0,343.40,65077
95827,fffcd46ef2263f404302a634eb57f7eb,5,1,84c5d4fbaf120aae381fad077416eaa0,66.083333,0,6,1,-9,1,1,0.00,386.53,0.0,0.0,0.0,386.53,81690
95828,fffce4705a9662cd70adb13d4a31832d,5,0,29309aa813182aaddc9b259e31b870e6,24.116667,1,1,1,-13,1,3,116.85,0.00,0.0,0.0,0.0,116.85,4039
95829,fffe18544ffabc95dfada21779c9644f,5,1,b5e6afd5a41800fdf401e0272ca74655,1.033333,0,1,1,-9,1,3,64.71,0.00,0.0,0.0,0.0,64.71,13289


In [40]:
df5=pd.merge(df4,order_item_df,on='order_id',how='inner')
df5

,order_id,review_score,review_answer_time,customer_id,order_approval_hours,carrier_delivered_days,customer_delivered_days,order_timely_delivered?,extra_days_taken_for_delivery,tot_orders,...,boleto,voucher,debit_card,others,total_amount,customer_zip_code_prefix,product_id,total_items,total_price,total_freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,5,1,3ce436f183e68e07877b285a838db11a,0.766667,6,1,1,-9,1,...,0.00,0.0,0.0,0.0,72.19,28013,4244733e06e7ecb4970a6e2683c13e61,1,58.900002,13.290000
1,00018f77f2f0320c557190d7a144bdd3,4,2,f6dd3ec061db4e3987629fe6b26e5cce,0.200000,8,8,1,-3,1,...,0.00,0.0,0.0,0.0,259.83,15775,e5f2d52b802189ee658865ca93d83a8f,1,239.899994,19.930000
2,000229ec398224ef6ca0657da4fc703e,5,0,6489ae5e4333f3693df5ad4372dab6d3,0.250000,1,6,1,-14,1,...,0.00,0.0,0.0,0.0,216.87,35661,c777355d18b72b67abbeef9df44fd0fd,1,199.000000,17.870001
3,00024acbcdf0a6daa1e931b038114c75,4,0,d4eb9395c8c0431ee92fce09860c5a06,0.166667,2,4,1,-6,1,...,0.00,0.0,0.0,0.0,25.78,12952,7634da152a4610f1595efa32f14722fc,1,12.990000,12.790000
4,00042b26cf59d7ce69dfabb4e55b4fd9,5,1,58dbd0b2d70206bf40e62cd34e84d795,0.216667,11,13,1,-16,1,...,0.00,0.0,0.0,0.0,218.04,13226,ac6c3623068f30de03045865e4e10089,1,199.899994,18.139999
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99497,fffc94f6ce00a00581880bf54a75a037,5,3,b51593916b4b8e0d6f66f2ae24f2673d,38.233333,0,15,1,-8,1,...,343.40,0.0,0.0,0.0,343.40,65077,4aa6014eceb682077f9dc4bffebc05b0,1,299.989990,43.410000
99498,fffcd46ef2263f404302a634eb57f7eb,5,1,84c5d4fbaf120aae381fad077416eaa0,66.083333,0,6,1,-9,1,...,386.53,0.0,0.0,0.0,386.53,81690,32e07fd915822b0765e448c4dd74c828,1,350.000000,36.529999
99499,fffce4705a9662cd70adb13d4a31832d,5,0,29309aa813182aaddc9b259e31b870e6,24.116667,1,1,1,-13,1,...,0.00,0.0,0.0,0.0,116.85,4039,72a30483855e2eafc67aee5dc2560482,1,99.900002,16.950001
99500,fffe18544ffabc95dfada21779c9644f,5,1,b5e6afd5a41800fdf401e0272ca74655,1.033333,0,1,1,-9,1,...,0.00,0.0,0.0,0.0,64.71,13289,9c422a519119dcad7575db5af1ba540e,1,55.990002,8.720000


In [41]:
products_df=products.drop(columns='product_category_name')

In [42]:
products_df

,product_id,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,00066f42aeeb9f3007548bb9d3f33c38,6,300.0,20.0,16.0,16.0
1,00088930e925c41fd95ebfe695fd2655,4,1225.0,55.0,10.0,26.0
2,0009406fd7479715e4bef61dd91f2462,2,300.0,45.0,15.0,35.0
3,000b8f95fcb9e0096488278317764d19,3,550.0,19.0,24.0,12.0
4,000d9be29b5207b54e86aa1b1ac54872,4,250.0,22.0,11.0,15.0
...,...,...,...,...,...,...
32946,fff6177642830a9a94a0f2cba5e476d1,1,1400.0,42.0,25.0,15.0
32947,fff81cc3158d2725c0655ab9ba0f712c,1,58.0,16.0,16.0,11.0
32948,fff9553ac224cec9d15d49f5a263411f,3,150.0,16.0,13.0,11.0
32949,fffdb2d0ec8d6a61f0a0a0db3f25b441,5,150.0,16.0,4.0,13.0


In [43]:
final_df=pd.merge(df5,products_df,on='product_id',how='inner')
final_df

,order_id,review_score,review_answer_time,customer_id,order_approval_hours,carrier_delivered_days,customer_delivered_days,order_timely_delivered?,extra_days_taken_for_delivery,tot_orders,...,customer_zip_code_prefix,product_id,total_items,total_price,total_freight_value,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,00010242fe8c5a6d1ba2dd792cb16214,5,1,3ce436f183e68e07877b285a838db11a,0.766667,6,1,1,-9,1,...,28013,4244733e06e7ecb4970a6e2683c13e61,1,58.900002,13.290000,4,650.0,28.0,9.0,14.0
1,00018f77f2f0320c557190d7a144bdd3,4,2,f6dd3ec061db4e3987629fe6b26e5cce,0.200000,8,8,1,-3,1,...,15775,e5f2d52b802189ee658865ca93d83a8f,1,239.899994,19.930000,2,30000.0,50.0,30.0,40.0
2,000229ec398224ef6ca0657da4fc703e,5,0,6489ae5e4333f3693df5ad4372dab6d3,0.250000,1,6,1,-14,1,...,35661,c777355d18b72b67abbeef9df44fd0fd,1,199.000000,17.870001,2,3050.0,33.0,13.0,33.0
3,00024acbcdf0a6daa1e931b038114c75,4,0,d4eb9395c8c0431ee92fce09860c5a06,0.166667,2,4,1,-6,1,...,12952,7634da152a4610f1595efa32f14722fc,1,12.990000,12.790000,1,200.0,16.0,10.0,15.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,5,1,58dbd0b2d70206bf40e62cd34e84d795,0.216667,11,13,1,-16,1,...,13226,ac6c3623068f30de03045865e4e10089,1,199.899994,18.139999,1,3750.0,35.0,40.0,30.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99481,fffc94f6ce00a00581880bf54a75a037,5,3,b51593916b4b8e0d6f66f2ae24f2673d,38.233333,0,15,1,-8,1,...,65077,4aa6014eceb682077f9dc4bffebc05b0,1,299.989990,43.410000,3,10150.0,89.0,15.0,40.0
99482,fffcd46ef2263f404302a634eb57f7eb,5,1,84c5d4fbaf120aae381fad077416eaa0,66.083333,0,6,1,-9,1,...,81690,32e07fd915822b0765e448c4dd74c828,1,350.000000,36.529999,1,8950.0,45.0,26.0,38.0
99483,fffce4705a9662cd70adb13d4a31832d,5,0,29309aa813182aaddc9b259e31b870e6,24.116667,1,1,1,-13,1,...,4039,72a30483855e2eafc67aee5dc2560482,1,99.900002,16.950001,1,967.0,21.0,24.0,19.0
99484,fffe18544ffabc95dfada21779c9644f,5,1,b5e6afd5a41800fdf401e0272ca74655,1.033333,0,1,1,-9,1,...,13289,9c422a519119dcad7575db5af1ba540e,1,55.990002,8.720000,1,100.0,20.0,20.0,20.0


### Drop unnecessary columns ###

In [44]:
final_df=final_df.drop(columns=['order_id','customer_id','product_id'])
final_df

,review_score,review_answer_time,order_approval_hours,carrier_delivered_days,customer_delivered_days,order_timely_delivered?,extra_days_taken_for_delivery,tot_orders,Total_installments,credit_card,...,total_amount,customer_zip_code_prefix,total_items,total_price,total_freight_value,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,5,1,0.766667,6,1,1,-9,1,2,72.19,...,72.19,28013,1,58.900002,13.290000,4,650.0,28.0,9.0,14.0
1,4,2,0.200000,8,8,1,-3,1,3,259.83,...,259.83,15775,1,239.899994,19.930000,2,30000.0,50.0,30.0,40.0
2,5,0,0.250000,1,6,1,-14,1,5,216.87,...,216.87,35661,1,199.000000,17.870001,2,3050.0,33.0,13.0,33.0
3,4,0,0.166667,2,4,1,-6,1,2,25.78,...,25.78,12952,1,12.990000,12.790000,1,200.0,16.0,10.0,15.0
4,5,1,0.216667,11,13,1,-16,1,3,218.04,...,218.04,13226,1,199.899994,18.139999,1,3750.0,35.0,40.0,30.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99481,5,3,38.233333,0,15,1,-8,1,1,0.00,...,343.40,65077,1,299.989990,43.410000,3,10150.0,89.0,15.0,40.0
99482,5,1,66.083333,0,6,1,-9,1,1,0.00,...,386.53,81690,1,350.000000,36.529999,1,8950.0,45.0,26.0,38.0
99483,5,0,24.116667,1,1,1,-13,1,3,116.85,...,116.85,4039,1,99.900002,16.950001,1,967.0,21.0,24.0,19.0
99484,5,1,1.033333,0,1,1,-9,1,3,64.71,...,64.71,13289,1,55.990002,8.720000,1,100.0,20.0,20.0,20.0


In [45]:
final_df.columns

Index(['review_score', 'review_answer_time', 'order_approval_hours',
       'carrier_delivered_days', 'customer_delivered_days',
       'order_timely_delivered?', 'extra_days_taken_for_delivery',
       'tot_orders', 'Total_installments', 'credit_card', 'boleto', 'voucher',
       'debit_card', 'others', 'total_amount', 'customer_zip_code_prefix',
       'total_items', 'total_price', 'total_freight_value',
       'product_photos_qty', 'product_weight_g', 'product_length_cm',
       'product_height_cm', 'product_width_cm'],
      dtype='str')

In [46]:
final_df.isna().sum()

review_score                     0
review_answer_time               0
order_approval_hours             0
carrier_delivered_days           0
customer_delivered_days          0
order_timely_delivered?          0
extra_days_taken_for_delivery    0
tot_orders                       0
Total_installments               0
credit_card                      0
boleto                           0
voucher                          0
debit_card                       0
others                           0
total_amount                     0
customer_zip_code_prefix         0
total_items                      0
total_price                      0
total_freight_value              0
product_photos_qty               0
product_weight_g                 0
product_length_cm                0
product_height_cm                0
product_width_cm                 0
dtype: int64

### Separate X and y variables ###

In [47]:
X=final_df[final_df.columns.difference(['review_score'])]
y=final_df['review_score']

In [48]:
X

,Total_installments,boleto,carrier_delivered_days,credit_card,customer_delivered_days,customer_zip_code_prefix,debit_card,extra_days_taken_for_delivery,order_approval_hours,order_timely_delivered?,...,product_photos_qty,product_weight_g,product_width_cm,review_answer_time,tot_orders,total_amount,total_freight_value,total_items,total_price,voucher
0,2,0.00,6,72.19,1,28013,0.0,-9,0.766667,1,...,4,650.0,14.0,1,1,72.19,13.290000,1,58.900002,0.0
1,3,0.00,8,259.83,8,15775,0.0,-3,0.200000,1,...,2,30000.0,40.0,2,1,259.83,19.930000,1,239.899994,0.0
2,5,0.00,1,216.87,6,35661,0.0,-14,0.250000,1,...,2,3050.0,33.0,0,1,216.87,17.870001,1,199.000000,0.0
3,2,0.00,2,25.78,4,12952,0.0,-6,0.166667,1,...,1,200.0,15.0,0,1,25.78,12.790000,1,12.990000,0.0
4,3,0.00,11,218.04,13,13226,0.0,-16,0.216667,1,...,1,3750.0,30.0,1,1,218.04,18.139999,1,199.899994,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99481,1,343.40,0,0.00,15,65077,0.0,-8,38.233333,1,...,3,10150.0,40.0,3,1,343.40,43.410000,1,299.989990,0.0
99482,1,386.53,0,0.00,6,81690,0.0,-9,66.083333,1,...,1,8950.0,38.0,1,1,386.53,36.529999,1,350.000000,0.0
99483,3,0.00,1,116.85,1,4039,0.0,-13,24.116667,1,...,1,967.0,19.0,0,1,116.85,16.950001,1,99.900002,0.0
99484,3,0.00,0,64.71,1,13289,0.0,-9,1.033333,1,...,1,100.0,20.0,1,1,64.71,8.720000,1,55.990002,0.0


In [49]:
y

0        5
1        4
2        5
3        4
4        5
        ..
99481    5
99482    5
99483    5
99484    5
99485    5
Name: review_score, Length: 99486, dtype: int64

In [50]:
# To avoid data contamination we will do train test split first

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.20,random_state=123,stratify=y)

In [51]:
X_train.shape

(79588, 23)

In [52]:
y_train.shape

(79588,)

In [53]:
X_test.shape

(19898, 23)

In [54]:
y_test.shape

(19898,)

### Outlier treatment ###

In [55]:
q25 = X_train.quantile(0.25)
q75 = X_train.quantile(0.75)

iqr = q75 - q25

lower_bound = q25 - 1.5 * iqr
upper_bound = q75 + 1.5 * iqr

X_train = X_train.clip(lower=lower_bound, upper=upper_bound, axis=1)
X_test = X_test.clip(lower=lower_bound, upper=upper_bound, axis=1)

### Standardization ###

In [56]:
min_max_scale=MinMaxScaler()
min_max_scale

,"feature_range feature_range: tuple (min, max), default=(0, 1)Desired range of transformed data.","(0, ...)"
,"copy copy: bool, default=TrueSet to False to perform inplace row normalization and avoid acopy (if the input is already a numpy array).",True
,"clip clip: bool, default=FalseSet to True to clip transformed values of held-out data toprovided `feature_range`.Since this parameter will clip values, `inverse_transform` may notbe able to restore the original data... note:: Setting `clip=True` does not prevent feature drift (a distribution shift between training and test data). The transformed values are clipped to the `feature_range`, which helps avoid unintended behavior in models sensitive to out-of-range inputs (e.g. linear models). Use with care, as clipping can distort the distribution of test data... versionadded:: 0.24",False


In [57]:
min_max_scale.fit_transform(X_train)

array([[0.11764706, 0.        , 0.        , ..., 0.        , 0.22826998,
        0.        ],
       [0.35294118, 0.        , 0.4       , ..., 0.        , 0.09760509,
        0.        ],
       [0.11764706, 0.        , 0.13333333, ..., 0.        , 0.64193604,
        0.        ],
       ...,
       [0.11764706, 0.        , 1.        , ..., 0.        , 0.1642941 ,
        0.        ],
       [0.58823529, 0.        , 0.        , ..., 0.        , 0.26477978,
        0.        ],
       [0.11764706, 0.        , 1.        , ..., 0.        , 0.05667393,
        0.        ]], shape=(79588, 23))

In [58]:
X_train_scaled=pd.DataFrame(min_max_scale.fit_transform(X_train),columns=X_train.columns)
X_test_scaled=pd.DataFrame(min_max_scale.transform(X_test),columns=X_test.columns)

In [59]:
X_test_scaled.head(3)

,Total_installments,boleto,carrier_delivered_days,credit_card,customer_delivered_days,customer_zip_code_prefix,debit_card,extra_days_taken_for_delivery,order_approval_hours,order_timely_delivered?,...,product_photos_qty,product_weight_g,product_width_cm,review_answer_time,tot_orders,total_amount,total_freight_value,total_items,total_price,voucher
0,0.117647,0.0,0.133333,0.000000,0.418605,0.893106,0.0,0.325,0.015327,0.0,...,0.166667,0.222222,0.623656,0.0,0.0,0.377272,0.414204,0.0,0.419159,0.0
1,0.117647,0.0,0.533333,0.172043,0.046512,0.045516,0.0,0.450,0.032977,0.0,...,0.166667,0.172840,0.301075,1.0,0.0,0.148131,0.263416,0.0,0.169620,0.0
2,0.352941,0.0,0.133333,0.333500,0.511628,0.707356,0.0,0.450,0.006038,0.0,...,0.166667,0.530864,0.236559,0.5,0.0,0.312864,0.561283,0.0,0.325071,0.0


In [60]:
X_train_scaled.head(3)

,Total_installments,boleto,carrier_delivered_days,credit_card,customer_delivered_days,customer_zip_code_prefix,debit_card,extra_days_taken_for_delivery,order_approval_hours,order_timely_delivered?,...,product_photos_qty,product_weight_g,product_width_cm,review_answer_time,tot_orders,total_amount,total_freight_value,total_items,total_price,voucher
0,0.117647,0.0,0.000000,0.000000,0.232558,0.124190,0.0,0.375,1.000000,0.0,...,0.5,1.000000,0.666667,0.166667,0.0,0.207081,0.346098,0.0,0.228270,0.0
1,0.352941,0.0,0.400000,0.104688,0.046512,0.035675,0.0,0.500,0.011147,0.0,...,1.0,0.074074,0.129032,0.166667,0.0,0.079409,0.195839,0.0,0.097605,0.0
2,0.117647,0.0,0.133333,0.596032,0.325581,0.884024,0.0,0.400,0.006038,0.0,...,0.5,0.185185,0.301075,0.166667,0.0,0.580724,0.538492,0.0,0.641936,0.0


In [61]:
y_train.value_counts()

review_score
5    46612
4    15503
1     8266
3     6649
2     2558
Name: count, dtype: int64

### SMOTE ###

In [62]:
smote=SMOTE(sampling_strategy='auto',k_neighbors=3,random_state=123)

In [63]:
X_train_scaled_smote,y_train_smote=smote.fit_resample(X_train_scaled,y_train)

### Logistic Regression ###

In [64]:
# Model 0: Baseline model 

logit=LogisticRegression()
logit

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default sol

In [65]:
logit.fit(X_train_scaled,y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default sol

In [66]:
logit.predict(X_train_scaled)

array([5, 5, 5, ..., 5, 5, 5], shape=(79588,))

In [67]:
print(classification_report(y_train,logit.predict(X_train_scaled)))

              precision    recall  f1-score   support

           1       0.54      0.30      0.39      8266
           2       0.00      0.00      0.00      2558
           3       0.00      0.00      0.00      6649
           4       0.00      0.00      0.00     15503
           5       0.61      0.98      0.75     46612

    accuracy                           0.61     79588
   macro avg       0.23      0.26      0.23     79588
weighted avg       0.41      0.61      0.48     79588



In [68]:
print(classification_report(y_test,logit.predict(X_test_scaled)))

              precision    recall  f1-score   support

           1       0.54      0.28      0.37      2066
           2       0.00      0.00      0.00       640
           3       0.00      0.00      0.00      1662
           4       0.00      0.00      0.00      3876
           5       0.61      0.98      0.75     11654

    accuracy                           0.60     19898
   macro avg       0.23      0.25      0.22     19898
weighted avg       0.41      0.60      0.48     19898



## RFE (Feature elimination) ##

In [69]:
rfe=RFE(DecisionTreeClassifier(),n_features_to_select=10)

In [70]:
rfe.fit(X_train_scaled,y_train)

,"estimator estimator: ``Estimator`` instanceA supervised learning estimator with a ``fit`` method that providesinformation about feature importance(e.g. `coef_`, `feature_importances_`).",DecisionTreeClassifier()
,"n_features_to_select n_features_to_select: int or float, default=NoneThe number of features to select. If `None`, half of the features areselected. If integer, the parameter is the absolute number of featuresto select. If float between 0 and 1, it is the fraction of features toselect... versionchanged:: 0.24 Added float values for fractions.",10
,"step step: int or float, default=1If greater than or equal to 1, then ``step`` corresponds to the(integer) number of features to remove at each iteration.If within (0.0, 1.0), then ``step`` corresponds to the percentage(rounded down) of features to remove at each iteration.",1
,"verbose verbose: int, default=0Controls verbosity of output.",0
,"importance_getter importance_getter: str or callable, default='auto'If 'auto', uses the feature importance either through a `coef_`or `feature_importances_` attributes of estimator.Also accepts a string that specifies an attribute name/pathfor extracting feature importance (implemented with `attrgetter`).For example, give `regressor_.coef_` in case of:class:`~sklearn.compose.TransformedTargetRegressor` or`named_steps.clf.feature_importances_` in case ofclass:`~sklearn.pipeline.Pipeline` with its last step named `clf`.If `callable`, overrides the default feature importance getter.The callable is passed with the fitted estimator and it shouldreturn importance for each feature... versionadded:: 0.24",'auto'
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only available when `estimator` is a classifier.","ndarray[int64](5,)","[1,2,3,4,5]"
estimator_ estimator_: ``Estimator`` instanceThe fitted estimator used to select features.,DecisionTreeClassifier,DecisionTreeClassifier()
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](23,)","['Total_installments','boleto','carrier_delivered_days',...,'total_items', 'total_price','voucher']"
n_features_ n_features_: intThe number of selected features.,int64,np.int64(10)
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 0.24,int,23


In [71]:
rfe.support_

array([False, False, False,  True, False,  True, False,  True,  True,
       False, False,  True,  True, False,  True, False, False, False,
        True,  True, False,  True, False])

In [72]:
X_train_scaled.columns[rfe.support_]

Index(['credit_card', 'customer_zip_code_prefix',
       'extra_days_taken_for_delivery', 'order_approval_hours',
       'product_height_cm', 'product_length_cm', 'product_weight_g',
       'total_amount', 'total_freight_value', 'total_price'],
      dtype='str')

In [73]:
rfe.ranking_

array([ 5, 14,  4,  1,  3,  1, 13,  1,  1, 12, 11,  1,  1,  7,  1,  2,  6,
       10,  1,  1,  9,  1,  8])

In [74]:
X_train_scaled_rfe=X_train_scaled[X_train_scaled.columns[rfe.support_]]
X_test_scaled_rfe=X_test_scaled[X_test_scaled.columns[rfe.support_]]

In [75]:
# Model 1: RFE selected columns

rfe_logit=LogisticRegression(random_state=123)

In [76]:
rfe_logit.fit(X_train_scaled_rfe,y_train)

,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",123
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solv

In [77]:
rfe_logit.predict(X_train_scaled_rfe)

array([5, 5, 5, ..., 5, 5, 5], shape=(79588,))

In [78]:
print(classification_report(y_train,rfe_logit.predict(X_train_scaled_rfe)))

              precision    recall  f1-score   support

           1       0.55      0.21      0.30      8266
           2       0.00      0.00      0.00      2558
           3       0.00      0.00      0.00      6649
           4       0.00      0.00      0.00     15503
           5       0.60      0.99      0.75     46612

    accuracy                           0.60     79588
   macro avg       0.23      0.24      0.21     79588
weighted avg       0.41      0.60      0.47     79588



In [79]:
print(classification_report(y_test,rfe_logit.predict(X_test_scaled_rfe)))

              precision    recall  f1-score   support

           1       0.56      0.20      0.30      2066
           2       0.00      0.00      0.00       640
           3       0.00      0.00      0.00      1662
           4       0.00      0.00      0.00      3876
           5       0.60      0.99      0.75     11654

    accuracy                           0.60     19898
   macro avg       0.23      0.24      0.21     19898
weighted avg       0.41      0.60      0.47     19898



### KNN Classifier ###

In [80]:
# Model 0: Baseline model

knn=KNeighborsClassifier()
knn

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Refer to the example entitled:ref:`sphx_glr_auto_examples_neighbors_plot_classification.py`showing the impact of the `weights` parameter on the decisionboundary.",'uniform'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this is equivalentto using manhattan_distance (l1), and euclidean_distance (l2) for p = 2.For arbitrary p, minkowski_distance (l_p) is used. This parameter is expectedto be positive.",2
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance<https://docs.scipy.org/doc/scipy/reference/spatial.distance.html>`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'minkowski'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.Doesn't affect :meth:`fit` method.",None


In [81]:
knn.fit(X_train_scaled,y_train)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Refer to the example entitled:ref:`sphx_glr_auto_examples_neighbors_plot_classification.py`showing the impact of the `weights` parameter on the decisionboundary.",'uniform'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this is equivalentto using manhattan_distance (l1), and euclidean_distance (l2) for p = 2.For arbitrary p, minkowski_distance (l_p) is used. This parameter is expectedto be positive.",2
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance<https://docs.scipy.org/doc/scipy/reference/spatial.distance.html>`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'minkowski'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.Doesn't affect :meth:`fit` method.",None
Name,Type,Value
"classes_ classes_: array of shape (n_classes,)Class labels known to the classifier","ndarray[int64](5,)","[1,2,3,4,5]"
"effective_metric_ effective_metric_: str or callbleThe distance metric used. It will be same as the `metric` parameteror a synonym of it, e.g. 'euclidean' if the `metric` parameter set to'minkowski' and `p` parameter set to 2.",str,'eu...an'


In [82]:
knn.predict(X_train_scaled)

array([5, 5, 5, ..., 1, 5, 5], shape=(79588,))

In [83]:
print(classification_report(y_train,knn.predict(X_train_scaled)))

              precision    recall  f1-score   support

           1       0.59      0.49      0.53      8266
           2       0.53      0.14      0.22      2558
           3       0.45      0.23      0.31      6649
           4       0.50      0.38      0.43     15503
           5       0.71      0.87      0.79     46612

    accuracy                           0.66     79588
   macro avg       0.56      0.42      0.46     79588
weighted avg       0.63      0.66      0.63     79588



In [84]:
print(classification_report(y_test,knn.predict(X_test_scaled)))

              precision    recall  f1-score   support

           1       0.40      0.30      0.34      2066
           2       0.10      0.03      0.04       640
           3       0.11      0.06      0.08      1662
           4       0.22      0.17      0.19      3876
           5       0.62      0.77      0.69     11654

    accuracy                           0.52     19898
   macro avg       0.29      0.26      0.27     19898
weighted avg       0.46      0.52      0.48     19898



In [85]:
# Model 1: Hyperparameter tuning

param={'n_neighbors':[2,3,4,5,6],'weights':['distance','uniform'],'leaf_size':[20,30,40,50]}

In [86]:
gcv_knn=GridSearchCV(KNeighborsClassifier(),param_grid=param,cv=5,verbose=True,n_jobs=-1)

In [87]:
gcv_knn.fit(X_train_scaled,y_train)

Fitting 5 folds for each of 40 candidates, totalling 200 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",KNeighborsClassifier()
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'leaf_size': [20, 30, ...], 'n_neighbors': [2, 3, ...], 'weights': ['distance', 'uniform']}"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",True
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_

In [88]:
gcv_knn.best_params_

{'leaf_size': 20, 'n_neighbors': 6, 'weights': 'distance'}

In [89]:
gcv_knn.predict(X_train_scaled)

array([5, 5, 5, ..., 1, 4, 5], shape=(79588,))

In [90]:
print(classification_report(y_train,gcv_knn.predict(X_train_scaled)))

              precision    recall  f1-score   support

           1       1.00      1.00      1.00      8266
           2       1.00      1.00      1.00      2558
           3       1.00      1.00      1.00      6649
           4       1.00      1.00      1.00     15503
           5       1.00      1.00      1.00     46612

    accuracy                           1.00     79588
   macro avg       1.00      1.00      1.00     79588
weighted avg       1.00      1.00      1.00     79588



In [91]:
print(classification_report(y_test,gcv_knn.predict(X_test_scaled)))

              precision    recall  f1-score   support

           1       0.50      0.27      0.35      2066
           2       0.24      0.05      0.08       640
           3       0.15      0.04      0.07      1662
           4       0.25      0.14      0.18      3876
           5       0.62      0.85      0.72     11654

    accuracy                           0.56     19898
   macro avg       0.35      0.27      0.28     19898
weighted avg       0.49      0.56      0.50     19898



In [92]:
# Model 2: RFE selected columns

param={'n_neighbors':[2,3,4,5,6],'weights':['distance','uniform'],'leaf_size':[20,30,40,50]}

In [93]:
rfe_knn=GridSearchCV(KNeighborsClassifier(),param_grid=param,cv=5,verbose=True,n_jobs=-1)

In [94]:
rfe_knn.fit(X_train_scaled_rfe,y_train)

Fitting 5 folds for each of 40 candidates, totalling 200 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",KNeighborsClassifier()
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'leaf_size': [20, 30, ...], 'n_neighbors': [2, 3, ...], 'weights': ['distance', 'uniform']}"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",True
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_

In [95]:
rfe_knn.best_params_

{'leaf_size': 20, 'n_neighbors': 6, 'weights': 'distance'}

In [96]:
rfe_knn.predict(X_train_scaled_rfe)

array([5, 5, 5, ..., 1, 4, 5], shape=(79588,))

In [97]:
print(classification_report(y_train,rfe_knn.predict(X_train_scaled_rfe)))

              precision    recall  f1-score   support

           1       1.00      1.00      1.00      8266
           2       1.00      1.00      1.00      2558
           3       1.00      1.00      1.00      6649
           4       1.00      1.00      1.00     15503
           5       1.00      1.00      1.00     46612

    accuracy                           1.00     79588
   macro avg       1.00      1.00      1.00     79588
weighted avg       1.00      1.00      1.00     79588



In [98]:
print(classification_report(y_test,rfe_knn.predict(X_test_scaled_rfe)))

              precision    recall  f1-score   support

           1       0.50      0.30      0.37      2066
           2       0.20      0.04      0.07       640
           3       0.15      0.05      0.07      1662
           4       0.23      0.14      0.17      3876
           5       0.62      0.84      0.72     11654

    accuracy                           0.56     19898
   macro avg       0.34      0.27      0.28     19898
weighted avg       0.48      0.56      0.50     19898



In [99]:
# Model 0: Baseline Model

ada=AdaBoostClassifier(random_state=123)

In [100]:
ada.fit(X_train_scaled,y_train)

,"random_state random_state: int, RandomState instance or None, default=NoneControls the random seed given at each `estimator` at eachboosting iteration.Thus, it is only used when `estimator` exposes a `random_state`.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",123
,"estimator estimator: object, default=NoneThe base estimator from which the boosted ensemble is built.Support for sample weighting is required, as well as proper``classes_`` and ``n_classes_`` attributes. If ``None``, thenthe base estimator is :class:`~sklearn.tree.DecisionTreeClassifier`initialized with `max_depth=1`... versionadded:: 1.2 `base_estimator` was renamed to `estimator`.",None
,"n_estimators n_estimators: int, default=50The maximum number of estimators at which boosting is terminated.In case of perfect fit, the learning procedure is stopped early.Values must be in the range `[1, inf)`.",50
,"learning_rate learning_rate: float, default=1.0Weight applied to each classifier at each boosting iteration. A higherlearning rate increases the contribution of each classifier. There isa trade-off between the `learning_rate` and `n_estimators` parameters.Values must be in the range `(0.0, inf)`.",1.0
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels.","ndarray[int64](5,)","[1,2,3,4,5]"
estimator_ estimator_: estimatorThe base estimator from which the ensemble is grown... versionadded:: 1.2 `base_estimator_` was renamed to `estimator_`.,DecisionTreeClassifier,DecisionTreeC...r(max_depth=1)
estimator_errors_ estimator_errors_: ndarray of floatsClassification error for each estimator in the boostedensemble.,"ndarray[float64](50,)","[0.39,0.59,0.72,...,0.79,0.79,0.79]"
estimator_weights_ estimator_weights_: ndarray of floatsWeights for each estimator in the boosted ensemble.,"ndarray[float64](50,)","[1.84,1.01,0.44,...,0.04,0.07,0.07]"
estimators_ estimators_: list of classifiersThe collection of fitted sub-estimators.,list,"[DecisionTreeC...ate=843828734), DecisionTreeC...ate=914636141), DecisionTreeC...te=1228959102), DecisionTreeC...te=1840268610), ...]"
"feature_importances_ feature_importances_: ndarray of shape (n_features,)The impurity-based feature importances if supported by the``estimator`` (when based on decision trees).Warning: impurity-based feature importances can be misleading forhigh cardinality features (many unique values). See:func:`sklearn.inspection.permutation_importance` as an alternative.","ndarray[float64](23,)","[0.01,0. ,0.04,...,0. ,0.07,0. ]"


In [101]:
ada.predict(X_train_scaled)

array([5, 5, 5, ..., 5, 5, 5], shape=(79588,))

In [102]:
print(classification_report(y_train,ada.predict(X_train_scaled)))

              precision    recall  f1-score   support

           1       0.63      0.32      0.42      8266
           2       0.00      0.00      0.00      2558
           3       0.00      0.00      0.00      6649
           4       0.00      0.00      0.00     15503
           5       0.61      0.99      0.76     46612

    accuracy                           0.61     79588
   macro avg       0.25      0.26      0.24     79588
weighted avg       0.42      0.61      0.49     79588



In [103]:
print(classification_report(y_test,ada.predict(X_test_scaled)))

              precision    recall  f1-score   support

           1       0.62      0.30      0.41      2066
           2       0.00      0.00      0.00       640
           3       0.00      0.00      0.00      1662
           4       0.00      0.00      0.00      3876
           5       0.61      0.99      0.76     11654

    accuracy                           0.61     19898
   macro avg       0.25      0.26      0.23     19898
weighted avg       0.42      0.61      0.48     19898



In [104]:
# Model 1: HyperParameter tuning

param={'n_estimators':[10,20,30,40,50],'learning_rate':[1.0,0.5,0.25,0.30]}

In [105]:
gcv_ada=GridSearchCV(AdaBoostClassifier(random_state=123),param_grid=param,cv=5,verbose=True,n_jobs=-1)

In [106]:
gcv_ada.fit(X_train_scaled,y_train)

Fitting 5 folds for each of 20 candidates, totalling 100 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",AdaBoostClass...dom_state=123)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'learning_rate': [1.0, 0.5, ...], 'n_estimators': [10, 20, ...]}"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",True
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see ho

In [107]:
gcv_ada.predict(X_train_scaled)

array([5, 5, 5, ..., 5, 5, 5], shape=(79588,))

In [108]:
print(classification_report(y_train,gcv_ada.predict(X_train_scaled)))

              precision    recall  f1-score   support

           1       0.63      0.32      0.42      8266
           2       0.00      0.00      0.00      2558
           3       0.00      0.00      0.00      6649
           4       0.00      0.00      0.00     15503
           5       0.61      0.99      0.76     46612

    accuracy                           0.61     79588
   macro avg       0.25      0.26      0.24     79588
weighted avg       0.42      0.61      0.49     79588



In [109]:
print(classification_report(y_test,gcv_ada.predict(X_test_scaled)))

              precision    recall  f1-score   support

           1       0.61      0.30      0.41      2066
           2       0.00      0.00      0.00       640
           3       0.00      0.00      0.00      1662
           4       0.00      0.00      0.00      3876
           5       0.61      0.99      0.75     11654

    accuracy                           0.61     19898
   macro avg       0.24      0.26      0.23     19898
weighted avg       0.42      0.61      0.48     19898



### Decision tree classifier ###

In [110]:
# Model 0: Baseline model

dtree=DecisionTreeClassifier(random_state=123)

In [111]:
dtree.fit(X_train_scaled,y_train)

,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",123
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples

In [112]:
dtree.predict(X_train_scaled)

array([5, 5, 5, ..., 1, 4, 5], shape=(79588,))

In [113]:
print(classification_report(y_train,dtree.predict(X_train_scaled)))

              precision    recall  f1-score   support

           1       1.00      1.00      1.00      8266
           2       1.00      1.00      1.00      2558
           3       1.00      1.00      1.00      6649
           4       1.00      1.00      1.00     15503
           5       1.00      1.00      1.00     46612

    accuracy                           1.00     79588
   macro avg       1.00      1.00      1.00     79588
weighted avg       1.00      1.00      1.00     79588



In [114]:
print(classification_report(y_test,dtree.predict(X_test_scaled)))

              precision    recall  f1-score   support

           1       0.31      0.33      0.32      2066
           2       0.08      0.08      0.08       640
           3       0.12      0.14      0.13      1662
           4       0.23      0.25      0.24      3876
           5       0.64      0.61      0.62     11654

    accuracy                           0.45     19898
   macro avg       0.28      0.28      0.28     19898
weighted avg       0.46      0.45      0.46     19898



In [115]:
# Model 1: Hyperparameter tuning

param={'max_depth':[2,3,4,5,6,7],'max_features':[1,2,3,4,5,6,7],'max_leaf_nodes':[2,3,4,5,6,7]}

In [116]:
dtree_gcv=GridSearchCV(DecisionTreeClassifier(random_state=123),param_grid=param,cv=5,verbose=True,n_jobs=-1)

In [117]:
dtree_gcv.fit(X_train_scaled,y_train)

Fitting 5 folds for each of 252 candidates, totalling 1260 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",DecisionTreeC...dom_state=123)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'max_depth': [2, 3, ...], 'max_features': [1, 2, ...], 'max_leaf_nodes': [2, 3, ...]}"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",True
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_searc

In [118]:
dtree_gcv.predict(X_train_scaled)

array([5, 5, 5, ..., 5, 5, 5], shape=(79588,))

In [119]:
print(classification_report(y_train,dtree_gcv.predict(X_train_scaled)))

              precision    recall  f1-score   support

           1       0.61      0.32      0.42      8266
           2       0.00      0.00      0.00      2558
           3       0.00      0.00      0.00      6649
           4       0.00      0.00      0.00     15503
           5       0.61      0.99      0.76     46612

    accuracy                           0.61     79588
   macro avg       0.24      0.26      0.24     79588
weighted avg       0.42      0.61      0.49     79588



In [120]:
print(classification_report(y_test,dtree_gcv.predict(X_test_scaled)))

              precision    recall  f1-score   support

           1       0.60      0.31      0.41      2066
           2       0.00      0.00      0.00       640
           3       0.00      0.00      0.00      1662
           4       0.00      0.00      0.00      3876
           5       0.61      0.99      0.76     11654

    accuracy                           0.61     19898
   macro avg       0.24      0.26      0.23     19898
weighted avg       0.42      0.61      0.49     19898



## Model is not improving due to heavy class imbalance, Even with balancing techniques like SMOTE, it did not improve. ##

### rfe_logit.pkl is the final model ###

In [ ]:
joblib.dump(rfe_logit, "logistic_review_model.pkl")